In [1]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine_similarity
from sklearn.cluster import KMeans
import time

model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:

sentences = [
    # Sports
    "The football team won the championship after a thrilling final match.",
    "The club lifted the league trophy following an exciting victory in the final.",
    "A talented striker scored two goals to secure the team's win.",
    "The basketball players practiced their shooting skills before the tournament.",
    "The coach changed the team's strategy during the second half.",

    # Technology
    "Artificial intelligence is helping businesses automate repetitive tasks.",
    "Companies are using machine learning to make routine work more efficient.",
    "Cloud computing allows users to store and access data over the internet.",
    "Cybersecurity protects computer systems from unauthorized access and attacks.",
    "Developers use programming languages to create websites and software applications.",

    # Cooking
    "The chef prepared a creamy pasta dish with fresh vegetables.",
    "Fresh vegetables were combined with pasta to create a rich and creamy meal.",
    "Baking bread requires accurate measurements and careful control of temperature.",
    "The soup was seasoned with herbs, spices, and a little salt.",
    "Grilling chicken gives it a smoky flavor and a crispy outer layer.",

    # Travel
    "Tourists explored the historic streets and museums of the old city.",
    "Visitors discovered ancient landmarks and cultural sites while walking through the city.",
    "Travelers booked a comfortable hotel near the airport for their overnight stay.",
    "A road trip through the mountains offered beautiful views of the countryside.",
    "Many people enjoy visiting coastal towns to relax beside the sea."
]

start_time = time.time()
embeddings = model.encode(sentences)
end_time = time.time()

generation_time = end_time - start_time
print("Embedding generation time:", generation_time, "seconds")
print("Embedding shape:", embeddings.shape)

Embedding generation time: 0.5959634780883789 seconds
Embedding shape: (20, 384)


In [4]:
total_words = sum(len(s.split()) for s in sentences)
estimated_tokens = total_words / 0.75  # rough word-to-token estimate
estimated_cost = (estimated_tokens / 1_000_000) * 0.02

print("Total words:", total_words)
print("Estimated tokens:", estimated_tokens)
print(f"Estimated OpenAI cost: ${estimated_cost:.8f}")

Total words: 219
Estimated tokens: 292.0
Estimated OpenAI cost: $0.00000584


In [5]:
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(sentences)

tfidf_similarity = sk_cosine_similarity(tfidf_matrix)
embedding_similarity = util.cos_sim(embeddings, embeddings).numpy()

print("TF-IDF similarity matrix shape:", tfidf_similarity.shape)
print("Embedding similarity matrix shape:", embedding_similarity.shape)

TF-IDF similarity matrix shape: (20, 20)
Embedding similarity matrix shape: (20, 20)


In [6]:
pairs_to_check = [(0, 1), (5, 6), (10, 11), (15, 16)]

for i, j in pairs_to_check:
    tfidf_score = tfidf_similarity[i][j]
    embed_score = embedding_similarity[i][j]
    print(f"Pair ({i},{j}):")
    print(f"  TF-IDF score:    {tfidf_score:.4f}")
    print(f"  Embedding score: {embed_score:.4f}")
    print(f"  Sentence {i}: {sentences[i]}")
    print(f"  Sentence {j}: {sentences[j]}")
    print()

Pair (0,1):
  TF-IDF score:    0.1095
  Embedding score: 0.6368
  Sentence 0: The football team won the championship after a thrilling final match.
  Sentence 1: The club lifted the league trophy following an exciting victory in the final.

Pair (5,6):
  TF-IDF score:    0.0000
  Embedding score: 0.5874
  Sentence 5: Artificial intelligence is helping businesses automate repetitive tasks.
  Sentence 6: Companies are using machine learning to make routine work more efficient.

Pair (10,11):
  TF-IDF score:    0.4780
  Embedding score: 0.8353
  Sentence 10: The chef prepared a creamy pasta dish with fresh vegetables.
  Sentence 11: Fresh vegetables were combined with pasta to create a rich and creamy meal.

Pair (15,16):
  TF-IDF score:    0.1065
  Embedding score: 0.6656
  Sentence 15: Tourists explored the historic streets and museums of the old city.
  Sentence 16: Visitors discovered ancient landmarks and cultural sites while walking through the city.



In [7]:
best_gaps = []
for i in range(20):
    for j in range(i+1, 20):
        gap = embedding_similarity[i][j] - tfidf_similarity[i][j]
        best_gaps.append((gap, i, j, tfidf_similarity[i][j], embedding_similarity[i][j]))

best_gaps.sort(reverse=True)

print("Top 8 biggest TF-IDF vs Embedding gaps:")
for gap, i, j, tf, emb in best_gaps[:8]:
    print(f"({i},{j}): TF-IDF={tf:.4f}, Embedding={emb:.4f}, Gap={gap:.4f}")
    print(f"  {sentences[i]}")
    print(f"  {sentences[j]}")
    print()

Top 8 biggest TF-IDF vs Embedding gaps:
(5,6): TF-IDF=0.0000, Embedding=0.5874, Gap=0.5874
  Artificial intelligence is helping businesses automate repetitive tasks.
  Companies are using machine learning to make routine work more efficient.

(15,16): TF-IDF=0.1065, Embedding=0.6656, Gap=0.5591
  Tourists explored the historic streets and museums of the old city.
  Visitors discovered ancient landmarks and cultural sites while walking through the city.

(0,1): TF-IDF=0.1095, Embedding=0.6368, Gap=0.5273
  The football team won the championship after a thrilling final match.
  The club lifted the league trophy following an exciting victory in the final.

(11,13): TF-IDF=0.0000, Embedding=0.4882, Gap=0.4882
  Fresh vegetables were combined with pasta to create a rich and creamy meal.
  The soup was seasoned with herbs, spices, and a little salt.

(1,2): TF-IDF=0.0000, Embedding=0.4452, Gap=0.4452
  The club lifted the league trophy following an exciting victory in the final.
  A talented

In [9]:
def embed_and_recommend(query_sentence, corpus_sentences, top_k=3):
    """
    Finds the most semantically similar sentences to a query using embeddings.
    Parameters:
        query_sentence (str) - the input sentence
        corpus_sentences (list of str) - sentences to search within
        top_k (int) - number of results to return
    Returns: list of (score, sentence) tuples, ranked highest first
    """
    query_embedding = model.encode(query_sentence)
    corpus_embeddings = model.encode(corpus_sentences)

    scores = util.cos_sim(query_embedding, corpus_embeddings)[0]

    top_results = np.argsort(scores.numpy())[::-1][:top_k]

    return [(scores[i].item(), corpus_sentences[i]) for i in top_results]


results = embed_and_recommend("The team celebrated their victory in the tournament.", sentences, top_k=3)
for score, sentence in results:
    print(f"{score:.4f} - {sentence}")

0.5342 - The football team won the championship after a thrilling final match.
0.5207 - The club lifted the league trophy following an exciting victory in the final.
0.4282 - The basketball players practiced their shooting skills before the tournament.


In [10]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

for i, (sentence, label) in enumerate(zip(sentences, cluster_labels)):
    print(f"Cluster {label}: [{i}] {sentence}")

Cluster 2: [0] The football team won the championship after a thrilling final match.
Cluster 2: [1] The club lifted the league trophy following an exciting victory in the final.
Cluster 2: [2] A talented striker scored two goals to secure the team's win.
Cluster 2: [3] The basketball players practiced their shooting skills before the tournament.
Cluster 2: [4] The coach changed the team's strategy during the second half.
Cluster 0: [5] Artificial intelligence is helping businesses automate repetitive tasks.
Cluster 0: [6] Companies are using machine learning to make routine work more efficient.
Cluster 0: [7] Cloud computing allows users to store and access data over the internet.
Cluster 0: [8] Cybersecurity protects computer systems from unauthorized access and attacks.
Cluster 0: [9] Developers use programming languages to create websites and software applications.
Cluster 3: [10] The chef prepared a creamy pasta dish with fresh vegetables.
Cluster 3: [11] Fresh vegetables were comb